# 02 — Kat-İçi Ön İşleme, Manuel/Otomatik Arama ve Nested Cross-Validation

**Yapay Sinir Ağları — Yüksek Lisans Ara Sınav Projesi** · Hakan Kızılhan

Bu notebook deneysel akışın (yönerge Bölüm 5) 4–9. adımlarını uygular. Kısaltmalar: CV (Cross-Validation, çapraz doğrulama), MLP (Multi-Layer Perceptron, çok katmanlı algılayıcı), AP (Average Precision, ortalama kesinlik), ROC-AUC (Receiver Operating Characteristic – Area Under Curve), SD (Standard Deviation, standart sapma), LR (Learning Rate, öğrenme oranı), BCE (Binary Cross-Entropy, ikili çapraz entropi).

| Bölüm | İşlem | Eğitim sayısı |
|---|---|---|
| A | Otomatik adayların (R1–R6) örneklenip kaydedilmesi → **commit noktası** | — |
| B1 | Outer `StratifiedKFold(3)` ve her outer-train içinde inner `StratifiedKFold(3)` | — |
| B2–B3 | 3 outer × 3 inner × (6 manuel + 6 otomatik) iç CV eğitimi; her aramanın seçtiği adayın outer-train'de yeniden eğitimi ve outer evaluation'da bir kez ölçülmesi; referans modeller (Dummy, Logistic Regression) | 108 + 6 |
| B4 | Outer CV sonuçları: kat bazlı ve Mean ± SD (n = 3) | — |
| B5 | Tüm development üzerinde iç CV ile nihai seçim | 36 |
| B6 | Ek analizler A1 (`unknown` → en sık sınıf) ve A2 (P1–P99 kırpma) — ana bütçe dışında | 6 |

Ortak kod (`ysa_core.py`): kat-içi ön işleme, PyTorch MLP, eğitim döngüsü (early stopping + ReduceLROnPlateau), metrikler. Nested döngünün kendisi bu notebook'ta açıkça yazılıdır.

**Her eğitimdeki üçlü veri ayrımı** (yönerge Bölüm 5): eğitim bölümü %85 *fit* (ön işleme parametreleri + ağırlık güncellemesi) ve %15 stratified *e-stop* (early stopping + LR scheduler, BCE kaybı) alt kümelerine ayrılır; adayın AP puanı ise eğitime hiç girmeyen ayrı değerlendirme katında hesaplanır.

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

import ysa_core as yc

ROOT = Path.cwd()
RES_DIR, FIG_DIR = ROOT / "results", ROOT / "figures"
cfgs = yc.load_configs(ROOT)
TRAIN, SEEDS = cfgs["training"], cfgs["seeds"]

yc.set_global_seed(SEEDS["global_python_numpy_torch"])
torch.set_num_threads(1)  # küçük MLP'lerde tek iş parçacığı hem daha hızlı hem tekrarlanabilir
yc.set_plot_style()
pd.set_option("display.max_columns", 50, "display.width", 180)

X, y = yc.load_development(ROOT, TRAIN["data"])
split = pd.read_csv(RES_DIR / "dataset_split.csv").set_index("orig_index")
print(f"Development: {X.shape[0]} satır × {X.shape[1]} özellik | pozitif: {y.sum()} (%{100 * y.mean():.2f})")
print(f"torch {torch.__version__} | iş parçacığı: {torch.get_num_threads()} | seed'ler: {SEEDS['base_seed']} tabanlı")


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    print(f"kaydedildi: figures/{name}.png")

## Bölüm A — Otomatik adayların (R1–R6) örneklenmesi

`ParameterSampler` ile her optimizer için tam 2 aday çekilir (seed = 31070 + optimizer sırası); LR log-ölçekte örneklenir (SGD: 1e-3–1e-1, Adam/AdamW: 1e-4–1e-2). Adaylar `configs/random_candidates.json` dosyasına yazılır. Dosya zaten varsa yeniden örnekleme yapılır ve kayıtlı adaylarla **birebir aynı olduğu doğrulanır** (adaylar sonradan değiştirilemez).

In [ ]:
MANUAL = [dict(c, unknown_strategy="category") for c in cfgs["manual_candidates"]["candidates"]]
RAND_PATH = ROOT / "configs" / "random_candidates.json"
sampled = yc.sample_random_candidates(cfgs["search_space"], SEEDS["random_search_sampler"])

if RAND_PATH.exists():
    assert yc.load_json(RAND_PATH)["candidates"] == sampled, "Kayıtlı adaylar yeniden örneklemeyle eşleşmiyor!"
    print("configs/random_candidates.json mevcut ve yeniden örneklemeyle birebir aynı ✓")
else:
    RAND_PATH.write_text(json.dumps({
        "_description": "Otomatik arama adayları (R1–R6): nested CV başlamadan önce örneklenip kaydedildi.",
        "sampler": "sklearn ParameterSampler, optimizer başına n_iter=2, random_state = 31070 + optimizer sırası",
        "candidates": sampled}, indent=2, ensure_ascii=False), encoding="utf-8")
    print("kaydedildi: configs/random_candidates.json")
RANDOM = sampled
BY_ID = {c["id"]: c for c in MANUAL + RANDOM}

PARAM_COLS = ["optimizer", "hidden_layers", "activation", "lr", "weight_decay", "batch_size", "scaler", "dropout",
              "clipping", "scheduler"]
candidates_table = pd.DataFrame(MANUAL + RANDOM).set_index("id")[PARAM_COLS]
candidates_table.insert(0, "arama", ["manuel"] * len(MANUAL) + ["otomatik"] * len(RANDOM))
candidates_table

**Not:** Önceden sabitlenmiş seed ile çekilen altı otomatik adayın hiçbirinde kırpma (`clipping`) seçilmemiştir (her aday için olasılık 1/2, altısında birden seçilmeme olasılığı 1/64). Örneklemeye müdahale edilmemiştir; kırpmanın etkisi kontrollü ek analiz A2 ile ayrıca sınanır.

---
### ⛔ Commit noktası

Bölüm B, otomatik adaylar `git` ile commit'lenmeden çalışmaz: aşağıdaki hücre `configs/random_candidates.json` dosyasının commit'lenmiş ve sonrasında değiştirilmemiş olduğunu kontrol eder. Böylece adayların outer sonuçlar görülmeden sabitlendiği zaman damgalı olarak belgelenir. (Repository dışında, ör. Colab'da çalıştırılırsa kontrol uyarıyla atlanır.)

In [ ]:
def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=ROOT)


if git("rev-parse", "--is-inside-work-tree").returncode != 0:
    print("UYARI: git repository'si bulunamadı — commit kontrolü atlandı")
else:
    last = git("log", "-1", "--format=%h %ad %s", "--date=iso", "--", "configs/random_candidates.json").stdout.strip()
    dirty = git("status", "--porcelain", "--", "configs/random_candidates.json").stdout.strip()
    assert last and not dirty, "Önce configs/random_candidates.json commit'lenmeli; Bölüm B ancak ondan sonra çalışır."
    print("Otomatik adayların commit'i:", last, "✓")

## Bölüm B1 — Outer ve inner katlar

Outer: `StratifiedKFold(n_splits=3, shuffle=True, random_state=31067)`. Inner: her outer-train içinde `StratifiedKFold(n_splits=3, shuffle=True, random_state=31068)`; manuel ve otomatik adaylar **aynı** inner katlarda değerlendirilir. Development içindeki 16 ikiz çiftten (01_eda, Bölüm 2) kaçının outer train/evaluation arasında bölündüğü de raporlanır.

In [ ]:
OUTER = list(StratifiedKFold(TRAIN["cv"]["outer"]["n_splits"], shuffle=True,
                             random_state=SEEDS["outer_cv"]).split(X, y))


def inner_folds(X_part, y_part):
    return list(StratifiedKFold(TRAIN["cv"]["inner"]["n_splits"], shuffle=True,
                                random_state=SEEDS["inner_cv"]).split(X_part, y_part))


twin_gid = split.loc[X.index, "group_id"].where(split.loc[X.index, "group_size"] > 1)
rows = []
for f, (tr, ev) in enumerate(OUTER, 1):
    inner_sizes = [len(a) for a, _ in inner_folds(X.iloc[tr], y.iloc[tr])]
    split_twins = len(set(twin_gid.iloc[tr].dropna()) & set(twin_gid.iloc[ev].dropna()))
    rows.append({"outer_fold": f, "outer_train": len(tr), "outer_train_pos": int(y.iloc[tr].sum()),
                 "outer_eval": len(ev), "outer_eval_pos": int(y.iloc[ev].sum()),
                 "inner_train (ort.)": np.mean(inner_sizes), "inner_val (ort.)": len(tr) - np.mean(inner_sizes),
                 "fit/e-stop (inner)": f"{round(np.mean(inner_sizes) * 0.85)}/{round(np.mean(inner_sizes) * 0.15)}",
                 "bölünen ikiz çift": split_twins})
fold_table = pd.DataFrame(rows).set_index("outer_fold")
fold_table.to_csv(RES_DIR / "cv_fold_sizes.csv")
fold_table

## Bölüm B2 — İç CV araması ve önceden tanımlanmış seçim kuralı

`inner_cv_search` bir aday listesini verilen eğitim bölümünün inner katlarında eğitir ve her katta AP hesaplar. `fit_candidate` (ysa_core) eğitim bölümünü %85 fit / %15 e-stop olarak ayırır ve ön işlemeyi **yalnızca fit alt kümesinde** öğrenir; değerlendirme katı bu fonksiyona hiç girmez.

**Seçim kuralı (sonuçlar görülmeden sabitlendi):** en yüksek iç CV mean AP; eşitlikte daha düşük SD; o da eşitse listedeki sıra.

Her eğitimin epoch günlüğü (train loss, e-stop loss, LR) ve özeti (en iyi/durulan epoch, süre, parametre sayısı, uyarılar) bellekte toplanır ve Bölüm B7'de CSV olarak yazılır.

In [ ]:
RUNS, HISTORIES = [], []


def train_and_log(cand, X_tr, y_tr, context, **tags):
    fit = yc.fit_candidate(cand, X_tr, y_tr, cfgs)
    RUNS.append({"context": context, **tags, "candidate": cand["id"], "optimizer": cand["optimizer"], **fit.info})
    HISTORIES.append(fit.history.assign(context=context, candidate=cand["id"], **tags))
    return fit


def inner_cv_search(cands, X_tr, y_tr, context, **tags):
    rows, folds = [], inner_folds(X_tr, y_tr)
    for cand in cands:
        for k, (a, b) in enumerate(folds, 1):
            fit = train_and_log(cand, X_tr.iloc[a], y_tr.iloc[a], context, inner_fold=k, **tags)
            m = yc.classification_metrics(y_tr.iloc[b], fit.predict_proba(X_tr.iloc[b]))
            rows.append({**tags, "candidate": cand["id"], "inner_fold": k, "AP": m["AP"], "ROC_AUC": m["ROC_AUC"],
                         "best_epoch": fit.info["best_epoch"], "stopped_epoch": fit.info["stopped_epoch"],
                         "n_params": fit.info["n_params"], "time_s": fit.info["total_time_s"],
                         "warnings": fit.info["warnings"]})
    return pd.DataFrame(rows)


def summarize_inner(df, cands):
    g = df.groupby("candidate")
    return pd.DataFrame({
        "AP_mean": g["AP"].mean(), "AP_sd": g["AP"].std(), "ROC_AUC_mean": g["ROC_AUC"].mean(),
        "best_epoch_mean": g["best_epoch"].mean(), "stopped_epoch_mean": g["stopped_epoch"].mean(),
        "n_params": g["n_params"].agg(lambda s: f"{s.min()}–{s.max()}" if s.min() != s.max() else str(s.min())),
        "time_total_s": g["time_s"].sum(),
        "warnings": g["warnings"].agg(lambda s: ";".join(sorted({w for x in s for w in x.split(";") if w}))),
    }).reindex([c["id"] for c in cands])


def select_best(summary):
    s = summary.reset_index().assign(order=range(len(summary)))
    return s.sort_values(["AP_mean", "AP_sd", "order"], ascending=[False, True, True]).iloc[0]["candidate"]


def reference_models(X_tr):
    return {"LogReg": Pipeline([("pre", yc.reference_preprocessor(X_tr)),
                                ("clf", LogisticRegression(max_iter=5000))]),
            "Dummy": DummyClassifier(strategy="prior")}

## Bölüm B3 — Nested CV

Her outer katta: (1) manuel arama (M1–M6) ve otomatik arama (R1–R6) aynı inner katlarda **ayrı ayrı** yürütülür; (2) her aramanın seçtiği aday outer-train'in tamamında (yine %85/%15 iç ayrımla) yeniden eğitilir; (3) outer evaluation üzerinde **yalnızca bir kez** ölçülür. Referans modeller (Dummy: sınıf önselini veren sabit tahmin; Logistic Regression: M1 ile aynı sabit ön işleme, varsayılan `C=1.0`) yalnızca outer-train ile eğitilir. Outer skorları hiçbir aday, parametre veya kuralı değiştirmek için kullanılmaz.

In [ ]:
CANDS = {"manual": MANUAL, "random": RANDOM}
inner_frames, outer_rows = [], []
t_start = time.perf_counter()

for f, (tr, ev) in enumerate(OUTER, 1):
    X_tr, y_tr, X_ev, y_ev = X.iloc[tr], y.iloc[tr], X.iloc[ev], y.iloc[ev]
    for search, cands in CANDS.items():
        t0 = time.perf_counter()
        df = inner_cv_search(cands, X_tr, y_tr, context="nested", outer_fold=f, search=search)
        summary = summarize_inner(df, cands)
        best = select_best(summary)
        inner_frames.append(df.assign(selected=df["candidate"] == best))
        refit = train_and_log(BY_ID[best], X_tr, y_tr, context="outer_refit", outer_fold=f, search=search)
        m = yc.classification_metrics(y_ev, refit.predict_proba(X_ev))
        outer_rows.append({"outer_fold": f, "method": f"MLP_{search}", "selected": best,
                           "inner_AP_mean": summary.loc[best, "AP_mean"], **m,
                           "best_epoch": refit.info["best_epoch"], "time_s": time.perf_counter() - t0})
        print(f"Outer {f} | {search:6s} → seçilen {best} (iç CV AP {summary.loc[best, 'AP_mean']:.4f}) "
              f"| outer AP {m['AP']:.4f} | {time.perf_counter() - t0:.0f} sn")
    for name, model in reference_models(X_tr).items():
        t0 = time.perf_counter()
        p = model.fit(X_tr, y_tr).predict_proba(X_ev)[:, 1]
        outer_rows.append({"outer_fold": f, "method": name, "selected": "-",
                           **yc.classification_metrics(y_ev, p), "time_s": time.perf_counter() - t0})

inner_all = pd.concat(inner_frames, ignore_index=True)
outer_df = pd.DataFrame(outer_rows)
inner_all.to_csv(RES_DIR / "inner_cv_fold_results.csv", index=False)
outer_df.to_csv(RES_DIR / "outer_cv_results.csv", index=False)
print(f"\nNested CV toplam süre: {time.perf_counter() - t_start:.0f} sn | MLP eğitim sayısı: {len(RUNS)}")

## Bölüm B4 — Outer CV sonuçları (Tablo 4 ve Tablo 5)

Kat bazlı ham sonuçlar ve üç outer kat üzerinden **ortalama ± örnek standart sapma (n = 3)**. Bu katlar birbirinden bağımsız tekrarlar değildir (eğitim bölümleri örtüşür); SD bir güven aralığı gibi yorumlanamaz ve n = 3 nedeniyle belirsizliği sınırlı biçimde temsil eder. Outer sonuçlar **arama prosedürlerinin** genelleme performansını tanımlar; nihai model seçimi için kullanılmaz (bkz. B5).

In [ ]:
METRICS = ["AP", "ROC_AUC", "F1", "Sensitivity", "Specificity", "Balanced_Accuracy", "Precision", "Accuracy"]
METHOD_ORDER = ["MLP_manual", "MLP_random", "LogReg", "Dummy"]

table4 = outer_df.set_index(["outer_fold", "method"])[["selected", *METRICS, "TP", "FP", "TN", "FN"]]
table4.round(4)

In [ ]:
agg = outer_df.groupby("method")[METRICS].agg(["mean", "std"]).reindex(METHOD_ORDER)
agg.to_csv(RES_DIR / "outer_cv_summary.csv")
table5 = pd.DataFrame({m: agg[m]["mean"].map("{:.3f}".format) + " ± " + agg[m]["std"].map("{:.3f}".format)
                       for m in METRICS})
table5.index = [yc.METHOD_LABELS[i] for i in table5.index]
table5

## Bölüm B5 — Tüm development üzerinde nihai seçim

Yönerge Bölüm 12.1 gereği her iki arama prosedürü tüm development verisinde 3 katlı iç CV ile yeniden çalıştırılır (outer sonuçlardan bağımsız). **Önceden tanımlanmış kural:** her arama kendi kazananını B2'deki kuralla seçer; iki kazanan arasından development iç CV mean AP'si yüksek olan nihai konfigürasyondur (eşitlikte düşük SD, o da eşitse manuel arama). Karşılaştırma amacıyla Logistic Regression ve Dummy de aynı iç katlarda değerlendirilir. Sınıflandırma eşiği varsayılan 0.50 olarak sabittir; eşik araması yapılmamıştır.

In [ ]:
t0 = time.perf_counter()
dev_frames = {s: inner_cv_search(c, X, y, context="dev_selection", outer_fold=0, search=s) for s, c in CANDS.items()}
dev_inner = pd.concat(dev_frames.values(), ignore_index=True)
dev_summary = pd.concat({s: summarize_inner(d, CANDS[s]) for s, d in dev_frames.items()}, names=["search", "candidate"])

ref_rows = []
for k, (a, b) in enumerate(inner_folds(X, y), 1):
    for name, model in reference_models(X.iloc[a]).items():
        p = model.fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1]
        ref_rows.append({"method": name, "inner_fold": k, **yc.classification_metrics(y.iloc[b], p)})
dev_ref = pd.DataFrame(ref_rows)

winners = pd.DataFrame([{"search": s, "candidate": select_best(dev_summary.loc[s]),
                         **dev_summary.loc[(s, select_best(dev_summary.loc[s])), ["AP_mean", "AP_sd"]]}
                        for s in CANDS])
final = winners.sort_values(["AP_mean", "AP_sd"], ascending=[False, True], kind="stable").iloc[0]
FINAL_CFG = BY_ID[final["candidate"]]

dev_inner.to_csv(RES_DIR / "dev_selection_fold_results.csv", index=False)
dev_summary.join(candidates_table.drop(columns="arama"), on="candidate").to_csv(RES_DIR / "dev_selection_results.csv")
dev_ref.to_csv(RES_DIR / "dev_selection_reference_models.csv", index=False)
(RES_DIR / "final_selection.json").write_text(json.dumps({
    "rule": "Her arama kendi kazananını iç CV mean AP ile seçer (eşitlikte düşük SD, sonra sıra); iki kazanan "
            "arasından development iç CV mean AP'si yüksek olan seçilir (eşitlikte düşük SD, sonra manuel).",
    "winners": winners.to_dict(orient="records"),
    "selected_search": final["search"], "selected_candidate": final["candidate"],
    "dev_inner_AP_mean": float(final["AP_mean"]), "dev_inner_AP_sd": float(final["AP_sd"]),
    "threshold": TRAIN["selection"]["threshold"], "config": FINAL_CFG}, indent=2, ensure_ascii=False),
    encoding="utf-8")

print(f"Süre: {time.perf_counter() - t0:.0f} sn")
print("Arama kazananları:")
print(winners.round(4).to_string(index=False))
print(f"\nNİHAİ KONFİGÜRASYON → {final['candidate']} ({final['search']}): {yc.describe_candidate(FINAL_CFG)}")
print("Referanslar (aynı iç katlar):", dev_ref.groupby("method")["AP"].agg(["mean", "std"]).round(4).to_dict("index"))

## Bölüm B6 — Ek analizler A1 ve A2 (ana bütçe dışında)

M1'den **tek bir faktör** değiştirilerek, B5 ile aynı development iç katlarında (eşleştirilmiş karşılaştırma):

- **A1** (H5): `"unknown"` → NaN → kat-içi en sık sınıf imputasyonu (M1'de ayrı kategori)
- **A2** (H2): kat-içi P1–P99 yüzdelik kırpma (M1'de kırpma yok)

Bu analizler nihai model seçimine dahil edilmez.

In [ ]:
M1 = BY_ID["M1"]
ABLATIONS = [dict(M1, id="A1", unknown_strategy="most_frequent", purpose="unknown → kat-içi en sık sınıf"),
             dict(M1, id="A2", clipping=True, purpose="kat-içi P1–P99 kırpma")]
abl = inner_cv_search(ABLATIONS, X, y, context="ablation", outer_fold=0, search="ablation")
m1_dev = dev_inner[(dev_inner["search"] == "manual") & (dev_inner["candidate"] == "M1")]
paired = pd.concat([m1_dev, abl]).pivot(index="candidate", columns="inner_fold", values="AP")
paired.columns = [f"AP kat {k}" for k in paired.columns]
paired["AP_mean"], paired["AP_sd"] = paired.mean(axis=1), paired.iloc[:, :3].std(axis=1)
paired["Δ AP (− M1, ort.)"] = paired["AP_mean"] - paired.loc["M1", "AP_mean"]
paired["Δ>0 olan kat"] = [(paired.loc[c].iloc[:3] > paired.loc["M1"].iloc[:3]).sum() for c in paired.index]
paired.to_csv(RES_DIR / "ablation_results.csv")
paired.round(4)

## Bölüm B7 — Eğitim kayıtları, süreler ve uyarılar

`results/training_runs.csv`: her MLP eğitiminin özeti; `results/training_history.csv`: her eğitimin epoch bazlı train loss, e-stop loss ve LR değerleri.

In [ ]:
runs = pd.DataFrame(RUNS)
hist = pd.concat(HISTORIES, ignore_index=True)
runs.to_csv(RES_DIR / "training_runs.csv", index=False)
hist.to_csv(RES_DIR / "training_history.csv", index=False)

print(f"Toplam MLP eğitimi: {len(runs)} →", runs["context"].value_counts().to_dict())
print("Uyarılar:", runs["warnings"].replace("", "yok").value_counts().to_dict())
opt_time = runs.groupby("optimizer").agg(eğitim=("candidate", "size"), süre_ort_s=("train_time_s", "mean"),
                                         süre_toplam_s=("train_time_s", "sum"), en_iyi_epoch_ort=("best_epoch", "mean"),
                                         durulan_epoch_ort=("stopped_epoch", "mean"), lr_düşüşü_ort=("n_lr_drops", "mean"))
opt_time.rename(index=yc.OPTIMIZER_NAMES).round(2)

## Bölüm B8 — Zorunlu tablolar: manuel (Tablo 2) ve otomatik (Tablo 3) arama sonuçları

Her hücre: iç CV mean AP ± SD (3 inner kat). ★ = o outer katta (veya development seçiminde) aramanın seçtiği aday. En iyi epoch ve süre, adayın tüm iç CV eğitimleri (3 outer × 3 + development 3 = 12 eğitim) üzerinden ortalama/toplamdır.

In [ ]:
def search_table(search, cands):
    cols = {}
    for f in range(1, 4):
        d = inner_all[(inner_all["search"] == search) & (inner_all["outer_fold"] == f)]
        s, sel = summarize_inner(d, cands), d.loc[d["selected"], "candidate"].iloc[0]
        cols[f"Outer {f}"] = [f"{s.loc[c, 'AP_mean']:.3f} ± {s.loc[c, 'AP_sd']:.3f}" + (" ★" if c == sel else "")
                              for c in s.index]
    s, sel = dev_summary.loc[search], winners.set_index("search").loc[search, "candidate"]
    cols["Development (nihai seçim)"] = [f"{s.loc[c, 'AP_mean']:.3f} ± {s.loc[c, 'AP_sd']:.3f}"
                                         + (" ★" if c == sel else "") for c in s.index]
    allrun = pd.concat([inner_all[inner_all["search"] == search], dev_inner[dev_inner["search"] == search]])
    g = allrun.groupby("candidate")
    out = pd.DataFrame(cols, index=s.index)
    out.insert(0, "konfigürasyon", [yc.describe_candidate(BY_ID[c]) for c in s.index])
    out["en iyi epoch (ort.)"] = g["best_epoch"].mean().round(1)
    out["süre (s, toplam)"] = g["time_s"].sum().round(1)
    out["parametre"] = s["n_params"]
    out["uyarılar"] = g["warnings"].agg(lambda x: ";".join(sorted({w for v in x for w in v.split(";") if w})))
    return out


table2, table3 = search_table("manual", MANUAL), search_table("random", RANDOM)
table2.to_csv(RES_DIR / "manual_search_results.csv")
table3.to_csv(RES_DIR / "random_search_results.csv")
table2

In [ ]:
table3

## Bölüm B9 — Şekiller

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.6), gridspec_kw={"width_ratios": [1.6, 1]})

# Sol: tüm development iç CV — 12 aday (mean ± SD) + referans çizgileri
ds = dev_summary.reset_index()
xs = np.arange(len(ds))
for search, key in [("manual", "MLP_manual"), ("random", "MLP_random")]:
    d = ds[ds["search"] == search]
    ax1.errorbar(xs[d.index], d["AP_mean"], yerr=d["AP_sd"], fmt="o", ms=8, capsize=4, lw=1.5,
                 color=yc.METHOD_COLORS[key], label=yc.METHOD_LABELS[key])
for name, ls in [("LogReg", "--"), ("Dummy", ":")]:
    v = dev_ref.loc[dev_ref["method"] == name, "AP"].mean()
    ax1.axhline(v, color=yc.METHOD_COLORS[name], ls=ls, lw=1.5, label=f"{yc.METHOD_LABELS[name]} ({v:.3f})")
fi = ds.index[(ds["candidate"] == final["candidate"])][0]
ax1.annotate("nihai", (xs[fi], ds.loc[fi, "AP_mean"] + ds.loc[fi, "AP_sd"]), xytext=(0, 10),
             textcoords="offset points", ha="center", fontsize=9, color=yc.INK)
ax1.set_xticks(xs, ds["candidate"])
ax1.set_ylabel("İç CV AP (ortalama ± SD)")
ax1.set_title("Tüm development iç CV: manuel ve otomatik adaylar")
ax1.legend(fontsize=8.5, loc="best", ncol=2)
ax1.grid(axis="x", visible=False)

# Sağ: outer CV AP — kat bazlı noktalar + ortalama ± SD
for i, m in enumerate(METHOD_ORDER):
    v = outer_df.loc[outer_df["method"] == m, "AP"].to_numpy()
    ax2.scatter(np.full(len(v), i) + np.array([-0.12, 0, 0.12]), v, s=36, color=yc.METHOD_COLORS[m], zorder=3)
    ax2.errorbar(i + 0.3, v.mean(), yerr=v.std(ddof=1), fmt="D", ms=6, capsize=4, color=yc.INK, zorder=4)
ax2.set_xticks(range(len(METHOD_ORDER)), [yc.METHOD_LABELS[m].replace(" (", "\n(") for m in METHOD_ORDER], fontsize=9)
ax2.set_ylabel("Outer evaluation AP")
ax2.set_title("Outer CV (3 kat): renkli = katlar, siyah = ort. ± SD")
ax2.grid(axis="x", visible=False)
fig.tight_layout()
savefig(fig, "search_comparison")
plt.show()

In [ ]:
SHOW = ["M1", "M3", "M5", "M6"]
h = hist[(hist["context"] == "dev_selection") & (hist["inner_fold"] == 1) & hist["candidate"].isin(SHOW)]
r = runs[(runs["context"] == "dev_selection") & (runs["inner_fold"] == 1)].set_index("candidate")
fig, axes = plt.subplots(2, 4, figsize=(15, 6), sharex="col", gridspec_kw={"height_ratios": [2, 1]})
for j, cid in enumerate(SHOW):
    d = h[h["candidate"] == cid]
    ax, axl = axes[0, j], axes[1, j]
    ax.plot(d["epoch"], d["train_loss"], color=yc.METHOD_COLORS["MLP_manual"], lw=2, label="train (fit) loss")
    ax.plot(d["epoch"], d["es_loss"], color=yc.METHOD_COLORS["LogReg"], lw=2, label="e-stop loss")
    be = r.loc[cid, "best_epoch"]
    ax.axvline(be, color=yc.INK, ls="--", lw=1)
    ax.text(be, ax.get_ylim()[1], f" en iyi: {be}", va="top", fontsize=8.5, color=yc.INK)
    ax.set_title(f"{cid}: {BY_ID[cid]['purpose'].split('(')[0].strip()}", fontsize=10.5)
    ax.set_ylabel("BCE" if j == 0 else "")
    axl.step(d["epoch"], d["lr"], where="post", color=yc.METHOD_COLORS["MLP_random"], lw=2)
    axl.set_yscale("log")
    axl.set_xlabel("epoch")
    axl.set_ylabel("LR" if j == 0 else "")
axes[0, 0].legend(fontsize=8.5)
fig.suptitle("Eğitim ve e-stop kaybı, epoch bazlı LR (tüm development, iç kat 1)", fontweight="bold", y=1.01)
fig.tight_layout()
savefig(fig, "training_curves_manual")
plt.show()

## Bu notebook'un çıktıları

| Dosya | İçerik |
|---|---|
| `configs/random_candidates.json` | Otomatik arama adayları R1–R6 (nested CV'den önce commit'lendi) |
| `results/cv_fold_sizes.csv` | Outer/inner kat boyutları, pozitif sayıları, bölünen ikiz çiftler |
| `results/inner_cv_fold_results.csv` | Nested CV'deki 108 iç eğitimin kat bazlı AP / ROC-AUC / epoch / süre değerleri |
| `results/manual_search_results.csv` | Tablo 2 — M1–M6 iç CV mean AP ± SD, seçimler, epoch, süre |
| `results/random_search_results.csv` | Tablo 3 — R1–R6 iç CV mean AP ± SD, seçimler, epoch, süre |
| `results/outer_cv_results.csv` | Tablo 4 — outer kat bazlı sonuçlar (MLP manuel/otomatik, LogReg, Dummy) |
| `results/outer_cv_summary.csv` | Tablo 5 — outer CV Mean ± SD (n = 3) |
| `results/dev_selection_*.csv`, `results/final_selection.json` | Tüm development iç CV ile nihai seçim |
| `results/ablation_results.csv` | Ek analizler A1 / A2 |
| `results/training_runs.csv`, `results/training_history.csv` | Tüm eğitimlerin özetleri ve epoch günlükleri |
| `figures/search_comparison.png`, `figures/training_curves_manual.png` | Arama karşılaştırması ve eğitim eğrileri |